# Select, filter and sort

**Video 6.2** | ShopKart project

**Goal:** Pick columns, filter rows with conditions and sort orders - the pandas versions of what you did in Excel and SQL.

### How to open in Colab
1. Go to colab.research.google.com → **File → Upload notebook** → choose this `.ipynb` file.
2. Click the **folder icon** on the left sidebar (Files panel) and drag in the data files: `orders_raw.csv`.
3. Run cells one by one with **Shift + Enter**.

> Files you upload to the Files panel disappear when the runtime resets. If you see `FileNotFoundError`, upload them again.

## Excel vs SQL vs pandas - same idea, three tools
| Task | Excel / Sheets | SQL (Module 4) | pandas |
|---|---|---|---|
| Pick columns | Hide other columns | `SELECT order_id, city FROM orders` | `orders[["order_id", "city"]]` |
| First rows | Scroll to top | `LIMIT 5` | `.head(5)` |
| Filter rows | Filter button → condition | `WHERE payment_mode = 'COD'` | `orders[orders["payment_mode"] == "COD"]` |
| Two conditions | Two filters | `WHERE a AND b` | `orders[(cond1) & (cond2)]` |
| One of a list | Tick several boxes | `WHERE city IN ('Pune','Delhi')` | `orders[orders["city"].isin(["Pune", "Delhi"])]` |
| Sort | Sort Z→A | `ORDER BY delivery_days DESC` | `.sort_values("delivery_days", ascending=False)` |
| Count rows | COUNTA / status bar | `SELECT COUNT(*)` | `len(df)` or `.shape[0]` |

We still use the **raw** file here on purpose - you will see why cleaning matters.

In [1]:
import pandas as pd
orders = pd.read_csv("orders_raw.csv")

## 1. Selecting columns

In [2]:
# One column -> a Series
orders["city"].head()

0      Delhi
1    Chennai
2      DELHI
3    Lucknow
4    Chennai
Name: city, dtype: str

In [3]:
# Several columns -> double brackets -> a DataFrame
orders[["order_id", "city", "payment_mode", "delivery_days"]].head()

,order_id,city,payment_mode,delivery_days
0,O10001,Delhi,COD,4
1,O10002,Chennai,Debit Card,4
2,O10003,DELHI,Debit Card,2
3,O10004,Lucknow,COD,4
4,O10005,Chennai,UPI,2


## 2. Selecting rows: loc (by label/condition) and iloc (by position)

In [4]:
print(orders.iloc[0])            # first row by position
orders.loc[0:2, ["order_id", "quantity"]]   # rows labelled 0 to 2 (included!), two columns

order_id             O10001
order_date       2025-01-01
customer_id           C0593
product_id             P048
quantity                  6
unit_price              362
discount                0.2
city                  Delhi
payment_mode            COD
delivery_days             4
rating                  2.0
returned                  N
Name: 0, dtype: object


,order_id,quantity
0,O10001,6
1,O10002,2
2,O10003,1


## 3. Filtering with a condition

In [5]:
cod = orders[orders["payment_mode"] == "COD"]
print(len(cod), "COD orders")
cod.head()

998 COD orders


,order_id,order_date,customer_id,product_id,quantity,unit_price,discount,city,payment_mode,delivery_days,rating,returned
0,O10001,2025-01-01,C0593,P048,6,362,0.20,Delhi,COD,4,2.0,N
3,O10004,2025-01-01,C0118,P026,2,2020,0.15,Lucknow,COD,4,4.0,Y
16,O10017,2025-01-02,C0421,P054,2,1459,0.00,Kolkata,COD,3,5.0,N
20,O10021,2025-01-02,C0724,P045,1,299,0.00,Chennai,COD,5,3.0,Y
24,O10025,2025-01-03,C0626,P053,2,607,0.05,Bengaluru,COD,4,2.0,Y


In [6]:
# Slow deliveries: 8 days or more
slow = orders[orders["delivery_days"] >= 8]
print(len(slow), "orders took 8+ days")

124 orders took 8+ days


In [7]:
# Two conditions: & means AND, | means OR. Each condition in its own brackets!
slow_cod = orders[(orders["delivery_days"] >= 8) & (orders["payment_mode"] == "COD")]
print(len(slow_cod), "slow COD orders")

22 slow COD orders


In [8]:
# Returned orders - notice the messy Yes/Y problem
print((orders["returned"] == "Y").sum(), "rows say Y")
print(orders["returned"].isin(["Y", "Yes"]).sum(), "rows say Y or Yes")

683 rows say Y
695 rows say Y or Yes


## 4. Why cleaning matters: filter for Delhi

In [9]:
exact = (orders["city"] == "Delhi").sum()
any_spelling = orders["city"].str.strip().str.lower().isin(["delhi", "new delhi"]).sum()
print("city == 'Delhi':", exact)
print("all Delhi spellings:", any_spelling)

city == 'Delhi': 715
all Delhi spellings: 813


An exact filter misses Delhi orders typed as `delhi`, `DELHI`, `Delhi ` or `New Delhi`. Any report built on this would under-count Delhi. Video 6.3 fixes this once for all.

## 5. Sorting

In [10]:
# Top 5 slowest deliveries
orders.sort_values("delivery_days", ascending=False)[["order_id", "city", "delivery_days", "rating"]].head()

,order_id,city,delivery_days,rating
3628,O13592,Pune,12,1.0
3981,O13940,Pune,11,NaN
1356,O11346,Pune,11,2.0
4583,O14535,Lucknow,10,3.0
4314,O14270,Pune,10,3.0


In [11]:
# Sort by two columns: city A-Z, then rating high to low
orders.sort_values(["city", "rating"], ascending=[True, False])[["order_id", "city", "rating"]].head()

,order_id,city,rating
265,O10265,Mumbai,5.0
306,O10305,Mumbai,5.0
1208,O11199,Mumbai,5.0
2974,O12949,Mumbai,5.0
3365,O13335,Mumbai,5.0


In [12]:
# Biggest quantities - our suspicious 99s float to the top
orders.nlargest(8, "quantity")[["order_id", "product_id", "quantity", "unit_price"]]

,order_id,product_id,quantity,unit_price
799,O10791,P032,99,1341
1821,O11810,P030,99,401
3425,O13394,P052,99,1266
3866,O13825,P038,99,893
216,O10216,P009,10,14999
3490,O13458,P009,10,"₹14,999"
4438,O14392,P009,10,14999
0,O10001,P048,6,362


Four rows with 99 and three with 10. In 6.3 we look at the products: the 10s are Smartphones (a real bulk order), the 99s are typos.

## Practice
1. How many orders have a rating of 1?
2. Show the 5 most recent-looking orders paid by Net Banking with delivery 2 days (any columns you like).

In [13]:
print((orders["rating"] == 1).sum())
orders[(orders["payment_mode"] == "Net Banking") & (orders["delivery_days"] == 2)].tail()

19


,order_id,order_date,customer_id,product_id,quantity,unit_price,discount,city,payment_mode,delivery_days,rating,returned
4804,O14752,2025-12-16,C0148,P046,1,466,0.15,Bangalore,Net Banking,2,4.0,N
4848,O14795,2025-12-17,C0019,P044,1,255,0.00,Mumbai,Net Banking,2,3.0,N
4876,O14822,2025-12-19,C0361,P020,1,2220,0.00,Bengaluru,Net Banking,2,4.0,N
4984,O14930,2025-12-25,C0675,P011,2,1148,0.10,Bengaluru,Net Banking,2,4.0,Y
5052,O14998,2025-12-28,C0753,P042,2,980,0.05,Hyderabad,Net Banking,2,5.0,N
